# CarePath DARAG — Stage 00: Setup & Config  `[CPU]`
Bootstrap the repo, pick a **profile** (`smoke` plumbing check / `full` ViMedCSS
run), mount Drive on Colab, and print the resolved paths. Every later stage reuses
this `init_stage` so run-sizes and artifact paths have one source of truth.

In [ ]:
# --- CarePath stage bootstrap (short by design) ---
import importlib.util, os, subprocess, sys
from pathlib import Path

def _find(start):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
    tok = os.environ.get('CAREPATH_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
    if tok and url.startswith('https://github.com/'):
        url = url.replace('https://', f'https://x-access-token:{tok}@')
    subprocess.run(['git', 'clone', url, '/content/carepath'], check=True)
    REPO = Path('/content/carepath')
assert REPO, 'Open this notebook from inside the CarePath repo.'
os.chdir(REPO); sys.path.insert(0, str(REPO / 'apps' / 'api'))

PROFILE = 'full'   # <<< set to 'full' for the real ViMedCSS run
from carepath.gec.notebook import init_stage
CTX = init_stage(PROFILE); P = CTX.paths; PROF = CTX.profile


In [ ]:
# Install the GEC training stack (idempotent; needed once per Colab runtime).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'])


In [ ]:
from carepath.gec import env
print(env.gpu_report())
print('dataset    ->', CTX.dataset)
print('datastore  ->', P.datastore)
print('real pairs ->', P.real_pairs)
print('adapters   ->', P.adapters)
